# Academic Stress Modeling

## Purpose

This notebook contains the machine-learning workflow for the student stress prototype. The exploratory analysis, statistical inference, and feature-policy rationale remain in `analysis.ipynb`.

The primary experiment estimates Low, Medium, or High stress from contextual circumstances. It is a portfolio prototype—not a diagnosis or a validated future-risk instrument.

## 1. Load the dataset

The path logic supports execution from either the repository root or the `notebooks/` directory. The raw CSV is read without modification.

In [6]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "StressLevelDataset.csv"
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
file_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

assert df.shape == (1_100, 21)
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

print(f"File: {DATA_PATH.name}")
print(f"SHA-256: {file_hash}")
print(f"Shape: {df.shape}")

File: StressLevelDataset.csv
SHA-256: 14a45e92708b0c063ad4ab04563aa8fd4e3fc27157fd282e1c0658dc5161faed
Shape: (1100, 21)


## 2. Feature policy

The main student-facing model uses contextual inputs. Mental-health history is optional and self-reported. Proximal symptoms are included only in a comparison experiment, while academic performance remains excluded because its timing is unclear.

In [7]:
main_context_features = [
    "study_load",
    "bullying",
    "social_support",
    "noise_level",
    "peer_pressure",
    "teacher_student_relationship",
    "living_conditions",
    "safety",
    "basic_needs",
    "extracurricular_activities",
]

optional_sensitive_features = ["mental_health_history"]

proximal_comparison_features = [
    "anxiety_level",
    "self_esteem",
    "depression",
    "headache",
    "blood_pressure",
    "sleep_quality",
    "breathing_problem",
    "future_career_concerns",
]

excluded_features = {
    "academic_performance": (
        "Timing is unclear and performance may be a consequence of stress."
    ),
}

target_column = "stress_level"
context_with_history_features = main_context_features + optional_sensitive_features
full_comparison_features = (
    main_context_features
    + optional_sensitive_features
    + proximal_comparison_features
)

policy_groups = [
    set(main_context_features),
    set(optional_sensitive_features),
    set(proximal_comparison_features),
    set(excluded_features),
]
assert all(group <= set(df.columns) for group in policy_groups)
assert not any(
    left & right
    for index, left in enumerate(policy_groups)
    for right in policy_groups[index + 1:]
)
assert target_column not in set().union(*policy_groups)

feature_policy_table = pd.DataFrame({
    "role": [
        "Main context model",
        "Optional sensitive input",
        "Proximal comparison only",
        "Excluded",
    ],
    "variables": [
        ", ".join(main_context_features),
        ", ".join(optional_sensitive_features),
        ", ".join(proximal_comparison_features),
        ", ".join(excluded_features),
    ],
})

feature_policy_table

,role,variables
0,Main context model,"study_load, bullying, social_support, noise_le..."
1,Optional sensitive input,mental_health_history
2,Proximal comparison only,"anxiety_level, self_esteem, depression, headac..."
3,Excluded,academic_performance


## 3. Evaluation design

The model-development workflow uses a stratified 80/20 split:

- **Training set:** preprocessing, five-fold cross-validation, and model selection
- **Test set:** one final evaluation after model selection

The primary metric is **High-stress recall**, because missing a student who is actually in the High-stress class is the most important error. Precision and class-balanced metrics remain necessary to detect excessive false alerts.

In [8]:
from sklearn.model_selection import StratifiedKFold, train_test_split

TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 20_260_827
CV_FOLDS = 5

stress_labels = {0: "Low", 1: "Medium", 2: "High"}
y = df[target_column].copy()
all_indices = df.index.to_numpy()

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE,
    stratify=y,
)
train_indices = np.sort(train_indices)
test_indices = np.sort(test_indices)

assert len(train_indices) == 880
assert len(test_indices) == 220
assert not set(train_indices) & set(test_indices)
assert set(train_indices) | set(test_indices) == set(all_indices)

y_train = y.loc[train_indices].copy()
y_test = y.loc[test_indices].copy()

experiment_features = {
    "context_only": main_context_features,
    "context_with_history": context_with_history_features,
    "full_comparison": full_comparison_features,
}

experiment_splits = {}
for experiment_name, feature_names in experiment_features.items():
    experiment_splits[experiment_name] = {
        "X_train": df.loc[train_indices, feature_names].copy(),
        "X_test": df.loc[test_indices, feature_names].copy(),
    }

X_train = experiment_splits["context_only"]["X_train"]
X_test = experiment_splits["context_only"]["X_test"]

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=SPLIT_RANDOM_STATE,
)

split_rows = []
for split_name, split_target in [("Training", y_train), ("Test", y_test)]:
    counts = split_target.value_counts().sort_index()
    for code_value, label in stress_labels.items():
        count = int(counts.get(code_value, 0))
        split_rows.append({
            "split": split_name,
            "stress_level": label,
            "count": count,
            "percent": round(count / len(split_target) * 100, 1),
        })

split_balance_table = pd.DataFrame(split_rows)
split_balance_table

,split,stress_level,count,percent
0,Training,Low,299,34.0
1,Training,Medium,286,32.5
2,Training,High,295,33.5
3,Test,Low,74,33.6
4,Test,Medium,72,32.7
5,Test,High,74,33.6


## 4. Leakage-safe preprocessing

Imputation and scaling are enclosed inside each model pipeline. During cross-validation, the transformations therefore learn parameters only from the current training fold—not from validation or test students.

In [9]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_preprocessor(feature_names):
    """Build preprocessing that learns parameters from training data only."""
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("scale", StandardScaler()),
    ])

    return ColumnTransformer(
        transformers=[("coded", numeric_steps, feature_names)],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def make_model_pipeline(estimator, feature_names):
    """Combine preprocessing and a model to prevent preprocessing leakage."""
    return Pipeline([
        ("preprocess", make_preprocessor(feature_names)),
        ("model", clone(estimator)),
    ])


preprocessing_validation = make_preprocessor(main_context_features)
X_train_prepared = preprocessing_validation.fit_transform(X_train)
X_test_prepared = preprocessing_validation.transform(X_test)

assert X_train_prepared.shape == (880, len(main_context_features))
assert X_test_prepared.shape == (220, len(main_context_features))
assert np.isfinite(X_train_prepared).all()
assert np.isfinite(X_test_prepared).all()

preprocessing_check = pd.DataFrame({
    "check": [
        "Training rows",
        "Test rows",
        "Main context features",
        "Training/test index overlap",
        "Missing values in main experiment",
        "Target included among predictors",
    ],
    "result": [
        len(X_train),
        len(X_test),
        X_train.shape[1],
        len(set(train_indices) & set(test_indices)),
        int(X_train.isna().sum().sum() + X_test.isna().sum().sum()),
        target_column in X_train.columns,
    ],
})

preprocessing_check

,check,result
0,Training rows,880
1,Test rows,220
2,Main context features,10
3,Training/test index overlap,0
4,Missing values in main experiment,0
5,Target included among predictors,False


In [10]:
from sklearn.metrics import f1_score, make_scorer, recall_score

HIGH_STRESS_LABEL = 2

SCORING = {
    "high_stress_recall": make_scorer(
        recall_score,
        labels=[HIGH_STRESS_LABEL],
        average="macro",
        zero_division=0,
    ),
    "macro_f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0,
    ),
    "balanced_accuracy": "balanced_accuracy",
}

evaluation_policy = pd.DataFrame({
    "role": ["Primary", "Supporting", "Supporting", "Diagnostic"],
    "metric": [
        "High-stress recall",
        "High-stress precision",
        "Macro F1 and balanced accuracy",
        "Confusion matrix and per-class metrics",
    ],
    "purpose": [
        "Prioritize finding students who are actually High stress",
        "Monitor false High-stress alerts",
        "Check performance across all three classes",
        "Show which classes are confused",
    ],
})

evaluation_policy

,role,metric,purpose
0,Primary,High-stress recall,Prioritize finding students who are actually H...
1,Supporting,High-stress precision,Monitor false High-stress alerts
2,Supporting,Macro F1 and balanced accuracy,Check performance across all three classes
3,Diagnostic,Confusion matrix and per-class metrics,Show which classes are confused


## 5. Preparation result

- The training set contains 880 students and the untouched test set contains 220.
- Class proportions are nearly identical across the two sets.
- All feature experiments reuse the same student indices for fair comparisons.
- Preprocessing is fitted only within the relevant training data.
- High-stress recall is primary, but it will never be interpreted without precision and class-balanced metrics.

The integer feature codes are treated as ordered numeric measurements. Because several code definitions remain only partially documented, this is an explicit limitation.


## 6. Baseline models

Three context-only baselines are compared using the same five stratified training folds. Every estimator is enclosed in the preprocessing pipeline, so imputation and scaling are refitted inside each fold.

- The Dummy classifier establishes the minimum benchmark.
- Logistic regression provides an interpretable linear baseline.
- A depth-three decision tree provides a small nonlinear baseline.

The held-out test set is not used in this comparison.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier

high_stress_precision_scorer = make_scorer(
    precision_score,
    labels=[HIGH_STRESS_LABEL],
    average="macro",
    zero_division=0,
)

baseline_scoring = {
    **SCORING,
    "high_stress_precision": high_stress_precision_scorer,
    "accuracy": "accuracy",
}

baseline_models = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(
        max_iter=2_000,
        random_state=SPLIT_RANDOM_STATE,
    ),
    "Decision tree (depth 3)": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        random_state=SPLIT_RANDOM_STATE,
    ),
}

result_rows = []
fold_rows = []

for model_name, estimator in baseline_models.items():
    pipeline = make_model_pipeline(estimator, main_context_features)
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=baseline_scoring,
        return_train_score=True,
    )

    for metric_name in baseline_scoring:
        validation_scores = scores[f"test_{metric_name}"]
        training_scores = scores[f"train_{metric_name}"]
        result_rows.append({
            "model": model_name,
            "metric": metric_name,
            "cv_mean": validation_scores.mean(),
            "cv_sd": validation_scores.std(ddof=1),
            "train_mean": training_scores.mean(),
            "train_cv_gap": training_scores.mean() - validation_scores.mean(),
        })

        for fold_number, score in enumerate(validation_scores, start=1):
            fold_rows.append({
                "model": model_name,
                "fold": fold_number,
                "metric": metric_name,
                "score": score,
            })

baseline_results = pd.DataFrame(result_rows)
baseline_fold_results = pd.DataFrame(fold_rows)

metric_labels = {
    "high_stress_recall": "High recall",
    "high_stress_precision": "High precision",
    "macro_f1": "Macro F1",
    "balanced_accuracy": "Balanced accuracy",
    "accuracy": "Accuracy",
}
model_order = list(baseline_models)

performance_values = (
    baseline_results
    .pivot(index="model", columns="metric", values="cv_mean")
    .reindex(model_order)
    .rename(columns=metric_labels)
)
recall_sd = (
    baseline_results
    .query("metric == 'high_stress_recall'")
    .set_index("model")["cv_sd"]
    .reindex(model_order)
    .rename("Recall SD")
)

baseline_summary = (
    performance_values[
        [
            "High recall",
            "High precision",
            "Macro F1",
            "Balanced accuracy",
            "Accuracy",
        ]
    ]
    .join(recall_sd)
    .reset_index()
    .rename(columns={"model": "Model"})
)

baseline_summary.round(3)

,Model,High recall,High precision,Macro F1,Balanced accuracy,Accuracy,Recall SD
0,Dummy (most frequent),0.000,0.000,0.169,0.333,0.340,0.000
1,Logistic regression,0.875,0.847,0.868,0.868,0.868,0.068
2,Decision tree (depth 3),0.841,0.920,0.872,0.872,0.872,0.065


In [12]:
recall_folds = (
    baseline_fold_results
    .query("metric == 'high_stress_recall'")
    .pivot(index="model", columns="fold", values="score")
    .reindex(model_order)
)

recall_stability = pd.DataFrame({
    "Model": model_order,
    "Minimum fold recall": recall_folds.min(axis=1).to_numpy(),
    "Maximum fold recall": recall_folds.max(axis=1).to_numpy(),
    "Recall range": (
        recall_folds.max(axis=1) - recall_folds.min(axis=1)
    ).to_numpy(),
})

recall_stability.round(3)

,Model,Minimum fold recall,Maximum fold recall,Recall range
0,Dummy (most frequent),0.000,0.000,0.000
1,Logistic regression,0.780,0.949,0.169
2,Decision tree (depth 3),0.763,0.932,0.169


### Baseline findings

- The most-frequent Dummy classifier predicts Low stress for every student and has zero High-stress recall.
- Logistic regression has the highest mean High-stress recall (`0.875`) and is the **provisional baseline leader** for the application objective.
- The decision tree has lower High-stress recall (`0.841`) but higher High-stress precision (`0.920`).
- Logistic regression and the decision tree have similar fold-to-fold recall variability (`SD = 0.068` and `0.065`). The small mean-recall difference does not establish a stable final ranking.
- No model is selected as final, and the test set remains untouched.

## 7. Current modeling results

- The modeling workflow uses a reproducible 880/220 stratified split.
- High-stress recall is primary; precision, macro F1, balanced accuracy, and class-level errors provide safeguards.
- Logistic regression is the current context-only baseline leader, with mean cross-validated High-stress recall of `0.875`.
- Baseline variability requires a more stable comparison before final model selection.

**Next phase:** compare additional models and class-weighting strategies using repeated stratified cross-validation, while keeping the test set untouched.